NAFLD / MASLD Model Training

**Dataset:** NHANES  
**Task:** Participant-level tabular baseline

The target must follow the approved NHANES liver-label definition. Exclude CAP and stiffness measurements used to derive the label.

# Stage 1: Environment Setup

In [ ]:
# ------------------ Stage 1: Setup ------------------

%pip install -q pandas numpy matplotlib seaborn scikit-learn scipy xgboost shap

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, ConfusionMatrixDisplay
)

RANDOM_STATE = 42

# Stage 2: NHANES Dataset Loading

In [ ]:
# Stage 2: Load Dataset

from google.colab import files
import pandas as pd

uploaded = files.upload()

datasets = {}

for name in uploaded:
    if name.lower().endswith(".xpt"):
        datasets[name.rsplit(".", 1)[0]] = pd.read_sas(
            name, format="xport"
        )

print("Loaded datasets:")

for name, data in datasets.items():
    print(name, data.shape)

Saving ALQ_J.XPT to ALQ_J.XPT
Saving BMX_J.XPT to BMX_J.XPT
Saving DEMO_J.XPT to DEMO_J.XPT
Saving DIQ_J.XPT to DIQ_J.XPT
Saving LUX_J.XPT to LUX_J.XPT
Saving MCQ_J.XPT to MCQ_J.XPT
Loaded datasets:
ALQ_J (5533, 10)
BMX_J (8704, 21)
DEMO_J (9254, 46)
DIQ_J (8897, 54)
LUX_J (6401, 14)
MCQ_J (8897, 76)


In [ ]:
# ------------------ Stage 2: Dataset Inspection ------------------

for name, data in datasets.items():
    print(f"\n{'='*60}")
    print(f"Dataset: {name}")
    print(f"Shape: {data.shape}")
    print(f"{'='*60}")
    print(data.columns.tolist())


Dataset: ALQ_J
Shape: (5533, 10)
['SEQN', 'ALQ111', 'ALQ121', 'ALQ130', 'ALQ142', 'ALQ270', 'ALQ280', 'ALQ290', 'ALQ151', 'ALQ170']

Dataset: BMX_J
Shape: (8704, 21)
['SEQN', 'BMDSTATS', 'BMXWT', 'BMIWT', 'BMXRECUM', 'BMIRECUM', 'BMXHEAD', 'BMIHEAD', 'BMXHT', 'BMIHT', 'BMXBMI', 'BMXLEG', 'BMILEG', 'BMXARML', 'BMIARML', 'BMXARMC', 'BMIARMC', 'BMXWAIST', 'BMIWAIST', 'BMXHIP', 'BMIHIP']

Dataset: DEMO_J
Shape: (9254, 46)
['SEQN', 'SDDSRVYR', 'RIDSTATR', 'RIAGENDR', 'RIDAGEYR', 'RIDAGEMN', 'RIDRETH1', 'RIDRETH3', 'RIDEXMON', 'RIDEXAGM', 'DMQMILIZ', 'DMQADFC', 'DMDBORN4', 'DMDCITZN', 'DMDYRSUS', 'DMDEDUC3', 'DMDEDUC2', 'DMDMARTL', 'RIDEXPRG', 'SIALANG', 'SIAPROXY', 'SIAINTRP', 'FIALANG', 'FIAPROXY', 'FIAINTRP', 'MIALANG', 'MIAPROXY', 'MIAINTRP', 'AIALANGA', 'DMDHHSIZ', 'DMDFMSIZ', 'DMDHHSZA', 'DMDHHSZB', 'DMDHHSZE', 'DMDHRGND', 'DMDHRAGZ', 'DMDHREDZ', 'DMDHRMAZ', 'DMDHSEDZ', 'WTINT2YR', 'WTMEC2YR', 'SDMVPSU', 'SDMVSTRA', 'INDHHIN2', 'INDFMIN2', 'INDFMPIR']

Dataset: DIQ_J
Shape: (8897, 54)

# Stage 3: Dataset Configuration

In [ ]:
# ------------------ Stage 3: Dataset Configuration ------------------

ID_COLUMN = "SEQN"

# NHANES liver measurement columns
CAP_COLUMN = "LUXCAPM"
STIFFNESS_COLUMN = "LUXSMED"

print("Participant ID:", ID_COLUMN)
print("CAP measurement:", CAP_COLUMN)
print("Liver stiffness measurement:", STIFFNESS_COLUMN)

Participant ID: SEQN
CAP measurement: LUXCAPM
Liver stiffness measurement: LUXSMED


# Stage 4: Dataset Inspection and Validation

In [ ]:
# ------------------ Stage 4: Dataset Inspection and Validation ------------------

for name in datasets:
    print(f"\n{name}")
    print("Shape:", datasets[name].shape)
    print("Duplicate participants:", datasets[name]["SEQN"].duplicated().sum())
    print("Missing participant IDs:", datasets[name]["SEQN"].isna().sum())

print("\nLiver measurement summary:")
print(datasets["LUX_J"][["LUXCAPM", "LUXSMED"]].describe())


ALQ_J
Shape: (5533, 10)
Duplicate participants: 0
Missing participant IDs: 0

BMX_J
Shape: (8704, 21)
Duplicate participants: 0
Missing participant IDs: 0

DEMO_J
Shape: (9254, 46)
Duplicate participants: 0
Missing participant IDs: 0

DIQ_J
Shape: (8897, 54)
Duplicate participants: 0
Missing participant IDs: 0

LUX_J
Shape: (6401, 14)
Duplicate participants: 0
Missing participant IDs: 0

MCQ_J
Shape: (8897, 76)
Duplicate participants: 0
Missing participant IDs: 0

Liver measurement summary:
           LUXCAPM      LUXSMED
count  5948.000000  5950.000000
mean    258.474950     5.896387
std      63.922516     5.080713
min     100.000000     1.600000
25%     212.000000     4.000000
50%     255.000000     4.900000
75%     303.000000     6.100000
max     400.000000    75.000000


In [ ]:
# ------------------ Stage 4: Liver Examination Quality Inspection ------------------

lux = datasets["LUX_J"]

print("Liver examination status:")
print(lux["LUAXSTAT"].value_counts(dropna=False).sort_index())

print("\nReasons for partial examinations:")
print(lux["LUARXNC"].value_counts(dropna=False).sort_index())

print("\nMeasurement availability by examination status:")
print(
    lux.groupby("LUAXSTAT")[["LUXCAPM", "LUXSMED"]]
    .agg(["count", "mean"])
)

print("\nMeasurement quality indicators:")
print(
    lux[["LUXCPIQR", "LUXSIQR", "LUXSIQRM"]]
    .describe()
)

Liver examination status:
LUAXSTAT
1.0    5494
2.0     493
3.0     258
4.0     156
Name: count, dtype: int64

Reasons for partial examinations:
LUARXNC
1.0     257
2.0     129
3.0     107
NaN    5908
Name: count, dtype: int64

Measurement availability by examination status:
         LUXCAPM             LUXSMED        
           count        mean   count    mean
LUAXSTAT                                    
1.0         5492  257.726875    5494  5.6751
2.0          456  267.484649     456  8.5625
3.0            0         NaN       0     NaN
4.0            0         NaN       0     NaN

Measurement quality indicators:
           LUXCPIQR       LUXSIQR      LUXSIQRM
count  5.940000e+03  5.957000e+03  5.942000e+03
mean   3.861178e+01  9.969783e-01  1.546289e+01
std    2.107645e+01  2.179355e+00  2.098514e+01
min    5.397605e-79  5.397605e-79  5.397605e-79
25%    2.500000e+01  4.000000e-01  9.300000e+00
50%    3.500000e+01  7.000000e-01  1.350000e+01
75%    4.800000e+01  1.000000e+00  1.8900

In [ ]:
# ------------------ Stage 4: Measurement Quality Inspection ------------------

lux = datasets["LUX_J"].copy()

# Check possible encoded missing values
quality_cols = ["LUXCPIQR", "LUXSIQR", "LUXSIQRM"]

missing_code = 5.397605e-79

for col in quality_cols:
    lux[col] = lux[col].replace(missing_code, np.nan)

print("Quality indicator missing values:")
print(lux[quality_cols].isna().sum())

print("\nCAP availability by examination status:")
print(
    lux.groupby("LUAXSTAT")["LUXCAPM"]
    .agg(["count", "size"])
)

print("\nLiver stiffness availability by examination status:")
print(
    lux.groupby("LUAXSTAT")["LUXSMED"]
    .agg(["count", "size"])
)

print("\nCAP IQR distribution:")
print(lux["LUXCPIQR"].describe())

print("\nCAP IQR missing values by examination status:")
print(lux.groupby("LUAXSTAT")["LUXCPIQR"].apply(
    lambda x: x.isna().sum()
))

Quality indicator missing values:
LUXCPIQR    461
LUXSIQR     444
LUXSIQRM    459
dtype: int64

CAP availability by examination status:
          count  size
LUAXSTAT             
1.0        5492  5494
2.0         456   493
3.0           0   258
4.0           0   156

Liver stiffness availability by examination status:
          count  size
LUAXSTAT             
1.0        5494  5494
2.0         456   493
3.0           0   258
4.0           0   156

CAP IQR distribution:
count    5.940000e+03
mean     3.861178e+01
std      2.107645e+01
min      5.397605e-79
25%      2.500000e+01
50%      3.500000e+01
75%      4.800000e+01
max      2.060000e+02
Name: LUXCPIQR, dtype: float64

CAP IQR missing values by examination status:
LUAXSTAT
1.0      2
2.0     45
3.0    258
4.0    156
Name: LUXCPIQR, dtype: int64


In [ ]:
# ------------------ Stage 4: Quality Indicator Cleaning ------------------

lux = datasets["LUX_J"].copy()

quality_cols = ["LUXCPIQR", "LUXSIQR", "LUXSIQRM"]

# Replace extremely small encoded missing values with NaN
for col in quality_cols:
    lux.loc[lux[col] < 1e-10, col] = np.nan

print("Corrected quality indicator summary:")
print(lux[quality_cols].describe())

print("\nMissing values after correction:")
print(lux[quality_cols].isna().sum())

print("\nCheck minimum values:")
print(lux[quality_cols].min())

Corrected quality indicator summary:
          LUXCPIQR      LUXSIQR     LUXSIQRM
count  5882.000000  5931.000000  5931.000000
mean     38.992520     1.001349    15.491570
std      20.826649     2.183125    20.994018
min       1.000000     0.100000     0.800000
25%      25.000000     0.400000     9.300000
50%      35.000000     0.700000    13.500000
75%      48.000000     1.000000    18.900000
max     206.000000    66.900000  1338.000000

Missing values after correction:
LUXCPIQR    519
LUXSIQR     470
LUXSIQRM    470
dtype: int64

Check minimum values:
LUXCPIQR    1.0
LUXSIQR     0.1
LUXSIQRM    0.8
dtype: float64


# Stage 5: Cohort Selection and Target Preparation

In [ ]:
# ------------------ Stage 5: Cohort Selection ------------------

demo = datasets["DEMO_J"]
lux = datasets["LUX_J"]

# Merge demographic and liver examination data
liver_data = pd.merge(
    demo[["SEQN", "RIDAGEYR", "RIAGENDR", "RIDRETH1"]],
    lux,
    on="SEQN",
    how="inner"
)

print("Merged participants:", liver_data.shape[0])

# Select adults aged 20 and above
adult_data = liver_data[
    liver_data["RIDAGEYR"] >= 20
].copy()

print("Participants aged 20+:", adult_data.shape[0])

# Select complete examinations with available CAP
eligible = adult_data[
    (adult_data["LUAXSTAT"] == 1) &
    (adult_data["LUXCAPM"].notna())
].copy()

print("Participants with complete exams and CAP:", eligible.shape[0])

print("\nCAP summary:")
print(eligible["LUXCAPM"].describe())

Merged participants: 6401
Participants aged 20+: 5265
Participants with complete exams and CAP: 4509

CAP summary:
count    4509.000000
mean      265.182524
std        62.327396
min       100.000000
25%       219.000000
50%       265.000000
75%       309.000000
max       400.000000
Name: LUXCAPM, dtype: float64


In [ ]:
# ------------------ Stage 5: Liver Measurement Analysis ------------------

print("CAP distribution:")
print(eligible["LUXCAPM"].describe())

print("\nLiver stiffness distribution:")
print(eligible["LUXSMED"].describe())

print("\nCAP missing values:")
print(eligible["LUXCAPM"].isna().sum())

print("\nLiver stiffness missing values:")
print(eligible["LUXSMED"].isna().sum())

print("\nCAP percentiles:")
print(eligible["LUXCAPM"].quantile(
    [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
))

CAP distribution:
count    4509.000000
mean      265.182524
std        62.327396
min       100.000000
25%       219.000000
50%       265.000000
75%       309.000000
max       400.000000
Name: LUXCAPM, dtype: float64

Liver stiffness distribution:
count    4509.000000
mean        5.805345
std         4.585185
min         1.600000
25%         4.100000
50%         5.000000
75%         6.200000
max        75.000000
Name: LUXSMED, dtype: float64

CAP missing values:
0

Liver stiffness missing values:
0

CAP percentiles:
0.01    122.08
0.05    166.00
0.25    219.00
0.50    265.00
0.75    309.00
0.95    372.00
0.99    400.00
Name: LUXCAPM, dtype: float64


In [ ]:
# ------------------ Stage 5: CAP-Based Steatosis Target ------------------

CAP_THRESHOLD = 302

eligible["steatosis_flag"] = (
    eligible["LUXCAPM"] >= CAP_THRESHOLD
).astype(int)

print("CAP threshold:", CAP_THRESHOLD, "dB/m")

print("\nSteatosis class distribution:")
print(eligible["steatosis_flag"].value_counts())

print("\nSteatosis class percentages:")
print(
    eligible["steatosis_flag"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nSteatosis prevalence:")
print(
    eligible["steatosis_flag"].mean() * 100,
    "%"
)

CAP threshold: 302 dB/m

Steatosis class distribution:
steatosis_flag
0    3222
1    1287
Name: count, dtype: int64

Steatosis class percentages:
steatosis_flag
0    71.46
1    28.54
Name: proportion, dtype: float64

Steatosis prevalence:
28.542914171656687 %


# Stage 6: Additional Metabolic Dataset Loading

In [ ]:
# ------------------ Stage 6: Additional Metabolic Dataset Loading ------------------

uploaded = files.upload()

for name in uploaded:
    if name.lower().endswith(".xpt"):
        datasets[name.rsplit(".", 1)[0]] = pd.read_sas(
            name, format="xport"
        )

for name in datasets:
    print(name, datasets[name].shape)

Saving BPQ_J.xpt to BPQ_J.xpt
Saving BPX_J.xpt to BPX_J.xpt
Saving GHB_J.xpt to GHB_J.xpt
Saving GLU_J.xpt to GLU_J.xpt
Saving HDL_J.xpt to HDL_J.xpt
Saving TRIGLY_J.xpt to TRIGLY_J.xpt
ALQ_J (5533, 10)
BMX_J (8704, 21)
DEMO_J (9254, 46)
DIQ_J (8897, 54)
LUX_J (6401, 14)
MCQ_J (8897, 76)
BPQ_J (6161, 11)
BPX_J (8704, 21)
GHB_J (6401, 2)
GLU_J (3036, 4)
HDL_J (7435, 3)
TRIGLY_J (3036, 10)


In [ ]:
# ------------------ Stage 6: Additional Dataset Inspection ------------------

additional_datasets = [
    "BPQ_J", "BPX_J", "GHB_J",
    "GLU_J", "HDL_J", "TRIGLY_J"
]

for name in additional_datasets:
    print("\n" + "=" * 60)
    print("Dataset:", name)
    print("Columns:", datasets[name].columns.tolist())
    print("=" * 60)


Dataset: BPQ_J
Columns: ['SEQN', 'BPQ020', 'BPQ030', 'BPD035', 'BPQ040A', 'BPQ050A', 'BPQ080', 'BPQ060', 'BPQ070', 'BPQ090D', 'BPQ100D']

Dataset: BPX_J
Columns: ['SEQN', 'PEASCCT1', 'BPXCHR', 'BPAARM', 'BPACSZ', 'BPXPLS', 'BPXPULS', 'BPXPTY', 'BPXML1', 'BPXSY1', 'BPXDI1', 'BPAEN1', 'BPXSY2', 'BPXDI2', 'BPAEN2', 'BPXSY3', 'BPXDI3', 'BPAEN3', 'BPXSY4', 'BPXDI4', 'BPAEN4']

Dataset: GHB_J
Columns: ['SEQN', 'LBXGH']

Dataset: GLU_J
Columns: ['SEQN', 'WTSAF2YR', 'LBXGLU', 'LBDGLUSI']

Dataset: HDL_J
Columns: ['SEQN', 'LBDHDD', 'LBDHDDSI']

Dataset: TRIGLY_J
Columns: ['SEQN', 'WTSAF2YR', 'LBXTR', 'LBDTRSI', 'LBDLDL', 'LBDLDLSI', 'LBDLDLM', 'LBDLDMSI', 'LBDLDLN', 'LBDLDNSI']


In [ ]:
# ------------------ Stage 6: Metabolic Variable Inspection ------------------

# Select relevant variables
metabolic_cols = {
    "GHB_J": ["LBXGH"],
    "GLU_J": ["LBXGLU", "WTSAF2YR"],
    "HDL_J": ["LBDHDD"],
    "TRIGLY_J": ["LBXTR", "WTSAF2YR"],
    "BPQ_J": ["BPQ020", "BPQ030", "BPQ040A", "BPQ050A"],
    "DIQ_J": ["DIQ010"],
    "BMX_J": ["BMXBMI", "BMXWAIST"]
}

for name, cols in metabolic_cols.items():
    print("\n" + "=" * 60)
    print("Dataset:", name)
    print("=" * 60)

    data = datasets[name][cols]

    print("\nDescriptive statistics:")
    print(data.describe(include="all"))

    print("\nMissing values:")
    print(data.isna().sum())

    for col in cols:
        if col not in ["WTSAF2YR"]:
            print(f"\n{col} value counts:")
            print(data[col].value_counts(dropna=False).head(15))


Dataset: GHB_J

Descriptive statistics:
             LBXGH
count  6045.000000
mean      5.769562
std       1.037838
min       3.800000
25%       5.200000
50%       5.500000
75%       5.900000
max      16.200000

Missing values:
LBXGH    356
dtype: int64

LBXGH value counts:
LBXGH
5.4    571
5.5    550
5.3    530
5.6    466
5.2    442
5.7    406
5.1    367
NaN    356
5.8    347
5.0    294
5.9    238
6.0    203
4.9    178
6.1    160
6.2    125
Name: count, dtype: int64

Dataset: GLU_J

Descriptive statistics:
            LBXGLU      WTSAF2YR
count  2891.000000  3.036000e+03
mean    111.803182  8.974191e+04
std      35.533147  1.094395e+05
min      47.000000  5.397605e-79
25%      96.000000  2.974831e+04
50%     102.000000  5.505728e+04
75%     113.000000  1.017417e+05
max     451.000000  9.441532e+05

Missing values:
LBXGLU      145
WTSAF2YR      0
dtype: int64

LBXGLU value counts:
LBXGLU
NaN      145
97.0     118
101.0    113
95.0     112
99.0     109
100.0    109
102.0    106
96.0   

In [ ]:
# ------------------ Stage 6: Blood Pressure Inspection ------------------

bpx = datasets["BPX_J"].copy()

systolic_cols = ["BPXSY1", "BPXSY2", "BPXSY3", "BPXSY4"]
diastolic_cols = ["BPXDI1", "BPXDI2", "BPXDI3", "BPXDI4"]

bpx["mean_systolic"] = bpx[systolic_cols].mean(axis=1)
bpx["mean_diastolic"] = bpx[diastolic_cols].mean(axis=1)

print("Mean systolic blood pressure:")
print(bpx["mean_systolic"].describe())

print("\nMean diastolic blood pressure:")
print(bpx["mean_diastolic"].describe())

print("\nMissing blood pressure measurements:")
print(bpx[["mean_systolic", "mean_diastolic"]].isna().sum())

Mean systolic blood pressure:
count    6717.000000
mean      121.684780
std        20.353961
min        72.666667
25%       106.666667
50%       118.000000
75%       132.666667
max       238.000000
Name: mean_systolic, dtype: float64

Mean diastolic blood pressure:
count    6.717000e+03
mean     6.832931e+01
std      1.577704e+01
min      5.397605e-79
25%      6.066667e+01
50%      6.933333e+01
75%      7.733333e+01
max      1.353333e+02
Name: mean_diastolic, dtype: float64

Missing blood pressure measurements:
mean_systolic     1987
mean_diastolic    1987
dtype: int64


In [ ]:
# ------------------ Stage 6: Blood Pressure and Weight Cleaning ------------------

# Clean blood pressure measurements
bpx = datasets["BPX_J"].copy()

bp_cols = [
    "BPXSY1", "BPXSY2", "BPXSY3", "BPXSY4",
    "BPXDI1", "BPXDI2", "BPXDI3", "BPXDI4"
]

for col in bp_cols:
    bpx.loc[bpx[col] < 1e-10, col] = np.nan

# Calculate average available readings
systolic_cols = ["BPXSY1", "BPXSY2", "BPXSY3", "BPXSY4"]
diastolic_cols = ["BPXDI1", "BPXDI2", "BPXDI3", "BPXDI4"]

bpx["mean_systolic"] = bpx[systolic_cols].mean(axis=1)
bpx["mean_diastolic"] = bpx[diastolic_cols].mean(axis=1)

print("Cleaned systolic BP:")
print(bpx["mean_systolic"].describe())

print("\nCleaned diastolic BP:")
print(bpx["mean_diastolic"].describe())

print("\nMissing BP values:")
print(bpx[["mean_systolic", "mean_diastolic"]].isna().sum())


# Clean fasting subsample weights
glu = datasets["GLU_J"].copy()
trigly = datasets["TRIGLY_J"].copy()

for data in [glu, trigly]:
    data.loc[data["WTSAF2YR"] < 1e-10, "WTSAF2YR"] = np.nan

print("\nGLU fasting weight missing:", glu["WTSAF2YR"].isna().sum())
print("TRIGLY fasting weight missing:", trigly["WTSAF2YR"].isna().sum())

Cleaned systolic BP:
count    6717.000000
mean      121.684780
std        20.353961
min        72.666667
25%       106.666667
50%       118.000000
75%       132.666667
max       238.000000
Name: mean_systolic, dtype: float64

Cleaned diastolic BP:
count    6636.000000
mean       69.540336
std        13.064739
min         8.000000
25%        61.333333
50%        70.000000
75%        77.333333
max       135.333333
Name: mean_diastolic, dtype: float64

Missing BP values:
mean_systolic     1987
mean_diastolic    2068
dtype: int64

GLU fasting weight missing: 325
TRIGLY fasting weight missing: 325


In [ ]:
# ------------------ Stage 6: Blood Pressure Validation ------------------

print("Lowest diastolic BP readings:")
print(
    bpx.loc[
        bpx["mean_diastolic"] < 30,
        ["SEQN"] + diastolic_cols + ["mean_diastolic"]
    ].head(20)
)

print("\nNumber of participants with diastolic BP below 30:")
print((bpx["mean_diastolic"] < 30).sum())

print("\nNumber of participants with systolic BP below diastolic BP:")
print(
    (bpx["mean_systolic"] < bpx["mean_diastolic"]).sum()
)

print("\nAvailable blood pressure readings:")
print(bpx[bp_cols].notna().sum())

Lowest diastolic BP readings:
          SEQN  BPXDI1  BPXDI2  BPXDI3  BPXDI4  mean_diastolic
37     93741.0     NaN    14.0    12.0    12.0       12.666667
80     93785.0     NaN    22.0    32.0     NaN       27.000000
94     93800.0     NaN     NaN    12.0     NaN       12.000000
945    94699.0    32.0    26.0    26.0     NaN       28.000000
2143   95967.0    14.0     6.0    16.0     NaN       12.000000
5379   99395.0     NaN     NaN    34.0    14.0       24.000000
5441   99462.0    24.0    46.0    14.0     NaN       28.000000
5974  100031.0    28.0    24.0     NaN    24.0       25.333333
6919  101042.0    34.0    14.0    16.0     NaN       21.333333
7156  101297.0    28.0    28.0    24.0     NaN       26.666667
7701  101873.0    32.0     2.0    10.0     NaN       14.666667
7864  102054.0     NaN     NaN     8.0     NaN        8.000000
8523  102767.0    26.0    28.0     NaN     NaN       27.000000

Number of participants with diastolic BP below 30:
13

Number of participants with syst

In [ ]:
# ------------------ Stage 6: Questionnaire Variable Validation ------------------

questionnaire_vars = {
    "DIQ_J": ["DIQ010"],
    "BPQ_J": ["BPQ020", "BPQ030", "BPQ040A", "BPQ050A"],
    "ALQ_J": [
        "ALQ111", "ALQ121", "ALQ130",
        "ALQ142", "ALQ270", "ALQ280",
        "ALQ290", "ALQ151", "ALQ170"
    ]
}

for name, cols in questionnaire_vars.items():
    print("\n" + "=" * 60)
    print("Dataset:", name)
    print("=" * 60)

    for col in cols:
        print(f"\n{col}:")
        print(datasets[name][col].value_counts(dropna=False).sort_index())


Dataset: DIQ_J

DIQ010:
DIQ010
1.0     893
2.0    7816
3.0     184
9.0       4
Name: count, dtype: int64

Dataset: BPQ_J

BPQ020:
BPQ020
1.0    2137
2.0    4014
9.0      10
Name: count, dtype: int64

BPQ030:
BPQ030
1.0    1718
2.0     408
9.0      11
NaN    4024
Name: count, dtype: int64

BPQ040A:
BPQ040A
1.0    1942
2.0     192
7.0       1
9.0       2
NaN    4024
Name: count, dtype: int64

BPQ050A:
BPQ050A
1.0    1650
2.0     292
NaN    4219
Name: count, dtype: int64

Dataset: ALQ_J

ALQ111:
ALQ111
1.0    4545
2.0     585
NaN     403
Name: count, dtype: int64

ALQ121:
ALQ121
5.397605e-79    1049
1.000000e+00     142
2.000000e+00     155
3.000000e+00     273
4.000000e+00     323
5.000000e+00     340
6.000000e+00     606
7.000000e+00     351
8.000000e+00     299
9.000000e+00     476
1.000000e+01     527
7.700000e+01       2
9.900000e+01       2
NaN              988
Name: count, dtype: int64

ALQ130:
ALQ130
1.0      1317
2.0      1040
3.0       479
4.0       247
5.0       130
6.0       

In [ ]:
# ------------------ Stage 6: Questionnaire Data Cleaning ------------------

# Create separate copies
diq = datasets["DIQ_J"].copy()
bpq = datasets["BPQ_J"].copy()
alq = datasets["ALQ_J"].copy()

# Clean diabetes responses
diq["DIQ010"] = diq["DIQ010"].replace({
    9: np.nan
})

# Clean blood pressure questionnaire responses
bpq_cols = ["BPQ020", "BPQ030", "BPQ040A", "BPQ050A"]

for col in bpq_cols:
    bpq[col] = bpq[col].replace({
        7: np.nan,
        9: np.nan
    })

# Convert encoded zero values in alcohol variables
alcohol_zero_cols = [
    "ALQ121", "ALQ142", "ALQ270",
    "ALQ280", "ALQ290", "ALQ170"
]

for col in alcohol_zero_cols:
    alq.loc[
        np.isclose(alq[col], 5.397605e-79),
        col
    ] = 0

# Replace refused and don't know responses
alq_special_codes = {
    "ALQ121": [77, 99],
    "ALQ130": [777, 999],
    "ALQ142": [77, 99],
    "ALQ270": [77, 99],
    "ALQ280": [77, 99],
    "ALQ290": [77, 99],
    "ALQ151": [7, 9],
    "ALQ170": [777, 999]
}

for col, codes in alq_special_codes.items():
    alq[col] = alq[col].replace(codes, np.nan)

print("Diabetes responses:")
print(diq["DIQ010"].value_counts(dropna=False).sort_index())

print("\nHypertension responses:")
for col in bpq_cols:
    print(f"\n{col}:")
    print(bpq[col].value_counts(dropna=False).sort_index())

print("\nAlcohol responses after cleaning:")
for col in alcohol_zero_cols:
    print(f"\n{col}:")
    print(alq[col].value_counts(dropna=False).sort_index())

Diabetes responses:
DIQ010
1.0     893
2.0    7816
3.0     184
NaN       4
Name: count, dtype: int64

Hypertension responses:

BPQ020:
BPQ020
1.0    2137
2.0    4014
NaN      10
Name: count, dtype: int64

BPQ030:
BPQ030
1.0    1718
2.0     408
NaN    4035
Name: count, dtype: int64

BPQ040A:
BPQ040A
1.0    1942
2.0     192
NaN    4027
Name: count, dtype: int64

BPQ050A:
BPQ050A
1.0    1650
2.0     292
NaN    4219
Name: count, dtype: int64

Alcohol responses after cleaning:

ALQ121:
ALQ121
0.0     1049
1.0      142
2.0      155
3.0      273
4.0      323
5.0      340
6.0      606
7.0      351
8.0      299
9.0      476
10.0     527
NaN      992
Name: count, dtype: int64

ALQ142:
ALQ142
0.0     2051
1.0       34
2.0       26
3.0       62
4.0      103
5.0      105
6.0      214
7.0      142
8.0       96
9.0      235
10.0     418
NaN     2047
Name: count, dtype: int64

ALQ270:
ALQ270
0.0      766
1.0       12
2.0       18
3.0       21
4.0       37
5.0       60
6.0       56
7.0       85
8.0    

# Stage 7: Dataset Merging and Integration

In [ ]:
# ------------------ Stage 7: Dataset Merging and Integration ------------------

# Start with the eligible liver cohort
masld_data = eligible.copy()

# Merge demographic and metabolic datasets
merge_datasets = {
    "BMX_J": datasets["BMX_J"],
    "DIQ_J": diq,
    "BPQ_J": bpq,
    "ALQ_J": alq,
    "GHB_J": datasets["GHB_J"],
    "GLU_J": glu,
    "HDL_J": datasets["HDL_J"],
    "TRIGLY_J": trigly
}

for name, data in merge_datasets.items():
    masld_data = masld_data.merge(
        data,
        on="SEQN",
        how="left"
    )
    print(f"{name} merged: {masld_data.shape}")

# Add cleaned average blood pressure measurements
masld_data = masld_data.merge(
    bpx[["SEQN", "mean_systolic", "mean_diastolic"]],
    on="SEQN",
    how="left"
)

print("\nFinal merged dataset shape:", masld_data.shape)
print("Unique participants:", masld_data["SEQN"].nunique())
print("Duplicate participants:", masld_data["SEQN"].duplicated().sum())

BMX_J merged: (4509, 38)
DIQ_J merged: (4509, 91)
BPQ_J merged: (4509, 101)
ALQ_J merged: (4509, 110)
GHB_J merged: (4509, 111)
GLU_J merged: (4509, 114)
HDL_J merged: (4509, 116)
TRIGLY_J merged: (4509, 125)

Final merged dataset shape: (4509, 127)
Unique participants: 4509
Duplicate participants: 0


# Stage 8: Missing Data and Cohort Assessment

In [ ]:
# ------------------ Stage 8: Check Merged Column Names ------------------

print("Fasting weight columns:")
print([
    col for col in masld_data.columns
    if "WTSAF" in col
])

print("\nColumns containing fasting measurements:")
print([
    col for col in masld_data.columns
    if col in ["LBXGLU", "LBXTR"]
])

Fasting weight columns:
['WTSAF2YR_x', 'WTSAF2YR_y']

Columns containing fasting measurements:
['LBXGLU', 'LBXTR']


In [ ]:
# ------------------ Stage 8: Missing Data and Cohort Assessment ------------------

assessment_cols = [
    "LUXCAPM",
    "BMXBMI", "BMXWAIST",
    "LBXGH", "LBXGLU",
    "LBDHDD", "LBXTR",
    "mean_systolic", "mean_diastolic",
    "DIQ010", "BPQ020", "BPQ040A", "BPQ050A",
    "ALQ121", "ALQ130", "ALQ142", "ALQ170",
    "WTSAF2YR_x", "WTSAF2YR_y"
]

missing_summary = pd.DataFrame({
    "Available": masld_data[assessment_cols].notna().sum(),
    "Missing": masld_data[assessment_cols].isna().sum(),
    "Missing_Percent": (
        masld_data[assessment_cols].isna().mean() * 100
    ).round(2)
})

print("Missing Data Summary:")
display(missing_summary)

print("\nFasting subsample coverage:")
print("Participants with fasting glucose:",
      masld_data["LBXGLU"].notna().sum())

print("Participants with triglycerides:",
      masld_data["LBXTR"].notna().sum())

print("Participants with both measurements:",
      masld_data[["LBXGLU", "LBXTR"]].notna().all(axis=1).sum())

print("\nFasting sample weight:")
print("GLU weight available:",
      masld_data["WTSAF2YR_x"].notna().sum())

print("TRIGLY weight available:",
      masld_data["WTSAF2YR_y"].notna().sum())

Missing Data Summary:


,Available,Missing,Missing_Percent
LUXCAPM,4509,0,0.00
BMXBMI,4471,38,0.84
BMXWAIST,4344,165,3.66
LBXGH,4359,150,3.33
LBXGLU,2164,2345,52.01
LBDHDD,4287,222,4.92
LBXTR,2124,2385,52.89
mean_systolic,4331,178,3.95
mean_diastolic,4321,188,4.17
DIQ010,4506,3,0.07



Fasting subsample coverage:
Participants with fasting glucose: 2164
Participants with triglycerides: 2124
Participants with both measurements: 2124

Fasting sample weight:
GLU weight available: 2084
TRIGLY weight available: 2084


# Stage 9: Metabolic Risk Indicator Construction

In [ ]:
# ------------------ Stage 9: BMI Risk Indicator ------------------

# Add detailed race/ethnicity information for Asian-specific BMI threshold
race_data = datasets["DEMO_J"][["SEQN", "RIDRETH3"]]

masld_data = masld_data.merge(
    race_data,
    on="SEQN",
    how="left"
)

# BMI threshold: 23 for non-Hispanic Asian participants, 25 otherwise
bmi_threshold = np.where(
    masld_data["RIDRETH3"] == 6,
    23,
    25
)

# Create BMI risk indicator
masld_data["bmi_risk"] = np.where(
    masld_data["BMXBMI"].isna(),
    np.nan,
    (masld_data["BMXBMI"] >= bmi_threshold).astype(int)
)

print("BMI Risk Indicator:")
print(masld_data["bmi_risk"].value_counts(dropna=False).sort_index())

print("\nBMI Risk Percentage:")
print(
    (masld_data["bmi_risk"].value_counts(normalize=True) * 100)
    .round(2)
)

BMI Risk Indicator:
bmi_risk
0.0    1064
1.0    3407
NaN      38
Name: count, dtype: int64

BMI Risk Percentage:
bmi_risk
1.0    76.2
0.0    23.8
Name: proportion, dtype: float64


In [ ]:
# ------------------ Stage 9: Waist Circumference Risk Indicator ------------------

# Male threshold: 94 cm
# Female threshold: 80 cm

waist_threshold = np.where(
    masld_data["RIAGENDR"] == 1,
    94,
    80
)

masld_data["waist_risk"] = np.where(
    masld_data["BMXWAIST"].isna(),
    np.nan,
    (masld_data["BMXWAIST"] >= waist_threshold).astype(int)
)

print("Waist Circumference Risk Indicator:")
print(masld_data["waist_risk"].value_counts(dropna=False).sort_index())

print("\nWaist Risk Percentage:")
print(
    (masld_data["waist_risk"].value_counts(normalize=True) * 100)
    .round(2)
)

Waist Circumference Risk Indicator:
waist_risk
0.0     984
1.0    3360
NaN     165
Name: count, dtype: int64

Waist Risk Percentage:
waist_risk
1.0    77.35
0.0    22.65
Name: proportion, dtype: float64


In [ ]:
# ------------------ Stage 9: Glycemic Risk Indicator ------------------

# Identify elevated glycemic measurements
hba1c_risk = masld_data["LBXGH"] >= 5.7
glucose_risk = masld_data["LBXGLU"] >= 100
diabetes_risk = masld_data["DIQ010"].isin([1, 3])

# Check whether at least one source is available
glycemic_data_available = (
    masld_data[["LBXGH", "LBXGLU", "DIQ010"]]
    .notna()
    .any(axis=1)
)

# Create combined glycemic risk indicator
masld_data["glycemic_risk"] = np.where(
    glycemic_data_available,
    (
        hba1c_risk.fillna(False) |
        glucose_risk.fillna(False) |
        diabetes_risk
    ).astype(int),
    np.nan
)

print("Glycemic Risk Indicator:")
print(
    masld_data["glycemic_risk"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nGlycemic Risk Percentage:")
print(
    (masld_data["glycemic_risk"].value_counts(normalize=True) * 100)
    .round(2)
)

Glycemic Risk Indicator:
glycemic_risk
0.0    1933
1.0    2576
Name: count, dtype: int64

Glycemic Risk Percentage:
glycemic_risk
1.0    57.13
0.0    42.87
Name: proportion, dtype: float64


In [ ]:
# ------------------ Stage 9: Blood Pressure Risk Indicator ------------------

# Identify elevated blood pressure
systolic_risk = masld_data["mean_systolic"] >= 130
diastolic_risk = masld_data["mean_diastolic"] >= 85

# Identify current antihypertensive medication use
medication_risk = masld_data["BPQ050A"] == 1

# Check whether blood pressure data is available
bp_data_available = (
    masld_data[
        ["mean_systolic", "mean_diastolic", "BPQ050A"]
    ].notna().any(axis=1)
)

# Create combined blood pressure risk indicator
masld_data["bp_risk"] = np.where(
    bp_data_available,
    (
        systolic_risk.fillna(False) |
        diastolic_risk.fillna(False) |
        medication_risk.fillna(False)
    ).astype(int),
    np.nan
)

print("Blood Pressure Risk Indicator:")
print(
    masld_data["bp_risk"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nBlood Pressure Risk Percentage:")
print(
    (masld_data["bp_risk"].value_counts(normalize=True) * 100)
    .round(2)
)

Blood Pressure Risk Indicator:
bp_risk
0.0    2111
1.0    2276
NaN     122
Name: count, dtype: int64

Blood Pressure Risk Percentage:
bp_risk
1.0    51.88
0.0    48.12
Name: proportion, dtype: float64


In [ ]:
# ------------------ Stage 9: Triglyceride Risk Indicator ------------------

masld_data["triglyceride_risk"] = np.where(
    masld_data["LBXTR"].isna(),
    np.nan,
    (masld_data["LBXTR"] >= 150).astype(int)
)

print("Triglyceride Risk Indicator:")
print(
    masld_data["triglyceride_risk"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nTriglyceride Risk Percentage:")
print(
    (masld_data["triglyceride_risk"].value_counts(normalize=True) * 100)
    .round(2)
)

Triglyceride Risk Indicator:
triglyceride_risk
0.0    1693
1.0     431
NaN    2385
Name: count, dtype: int64

Triglyceride Risk Percentage:
triglyceride_risk
0.0    79.71
1.0    20.29
Name: proportion, dtype: float64


In [ ]:
# ------------------ Stage 9: HDL Cholesterol Risk Indicator ------------------

# Identify low HDL cholesterol using sex-specific thresholds
hdl_threshold = np.where(
    masld_data["RIAGENDR"] == 1,
    40,
    50
)

masld_data["hdl_risk"] = np.where(
    masld_data["LBDHDD"].isna(),
    np.nan,
    (masld_data["LBDHDD"] <= hdl_threshold).astype(int)
)

print("HDL Cholesterol Risk Indicator:")
print(
    masld_data["hdl_risk"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nHDL Risk Percentage:")
print(
    (masld_data["hdl_risk"].value_counts(normalize=True) * 100)
    .round(2)
)

HDL Cholesterol Risk Indicator:
hdl_risk
0.0    2909
1.0    1378
NaN     222
Name: count, dtype: int64

HDL Risk Percentage:
hdl_risk
0.0    67.86
1.0    32.14
Name: proportion, dtype: float64


In [ ]:
# ------------------ Stage 9: Metabolic Risk Indicator Summary ------------------

risk_cols = [
    "bmi_risk",
    "waist_risk",
    "glycemic_risk",
    "bp_risk",
    "triglyceride_risk",
    "hdl_risk"
]

risk_summary = pd.DataFrame({
    "Risk_Present": (masld_data[risk_cols] == 1).sum(),
    "Risk_Absent": (masld_data[risk_cols] == 0).sum(),
    "Missing": masld_data[risk_cols].isna().sum()
})

risk_summary["Risk_Percentage"] = (
    risk_summary["Risk_Present"] /
    (risk_summary["Risk_Present"] + risk_summary["Risk_Absent"]) * 100
).round(2)

print("Metabolic Risk Indicator Summary:")
display(risk_summary)

Metabolic Risk Indicator Summary:


,Risk_Present,Risk_Absent,Missing,Risk_Percentage
bmi_risk,3407,1064,38,76.20
waist_risk,3360,984,165,77.35
glycemic_risk,2576,1933,0,57.13
bp_risk,2276,2111,122,51.88
triglyceride_risk,431,1693,2385,20.29
hdl_risk,1378,2909,222,32.14


In [ ]:
# ------------------ Stage 9: Alcohol Exposure Assessment ------------------

# Approximate drinking occasions per week
frequency_per_week = {
    0: 0,
    1: 7,
    2: 6,
    3: 3.5,
    4: 2,
    5: 1,
    6: 2.5 / 4.345,
    7: 1 / 4.345,
    8: 9 / 52,
    9: 4.5 / 52,
    10: 1.5 / 52
}

# Convert drinking frequency into weekly occasions
drinking_frequency = masld_data["ALQ121"].map(
    frequency_per_week
)

# Estimate drinks consumed per week
weekly_drinks = drinking_frequency * masld_data["ALQ130"]

# Non-drinkers in the past year have zero alcohol consumption
weekly_drinks = weekly_drinks.where(
    masld_data["ALQ121"] != 0,
    0
)

# Estimate grams of alcohol per week
masld_data["alcohol_g_week"] = weekly_drinks * 14

# Sex-specific alcohol thresholds
alcohol_threshold = np.where(
    masld_data["RIAGENDR"] == 1,
    210,
    140
)

# Classify approximate alcohol eligibility
masld_data["alcohol_eligible"] = np.where(
    masld_data["alcohol_g_week"].isna(),
    np.nan,
    (
        masld_data["alcohol_g_week"] < alcohol_threshold
    ).astype(int)
)

print("Estimated Alcohol Exposure:")
print(
    masld_data["alcohol_g_week"]
    .describe()
    .round(2)
)

print("\nAlcohol Eligibility:")
print(
    masld_data["alcohol_eligible"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nAlcohol Eligibility Percentage:")
print(
    (
        masld_data["alcohol_eligible"]
        .value_counts(normalize=True) * 100
    ).round(2)
)

Estimated Alcohol Exposure:
count    3820.00
mean       44.27
std       112.03
min         0.00
25%         0.40
50%         4.85
75%        32.22
max      1470.00
Name: alcohol_g_week, dtype: float64

Alcohol Eligibility:
alcohol_eligible
0.0     243
1.0    3577
NaN     689
Name: count, dtype: int64

Alcohol Eligibility Percentage:
alcohol_eligible
1.0    93.64
0.0     6.36
Name: proportion, dtype: float64


In [ ]:
# ------------------ Stage 9: Alcohol Classification Review ------------------

male = masld_data["RIAGENDR"] == 1
female = masld_data["RIAGENDR"] == 2

masld_data["alcohol_category"] = np.select(
    [
        male & (masld_data["alcohol_g_week"] < 210),
        male & (masld_data["alcohol_g_week"].between(210, 420, inclusive="both")),
        male & (masld_data["alcohol_g_week"] > 420),

        female & (masld_data["alcohol_g_week"] < 140),
        female & (masld_data["alcohol_g_week"].between(140, 350, inclusive="both")),
        female & (masld_data["alcohol_g_week"] > 350)
    ],
    [
        "MASLD alcohol range",
        "MetALD range",
        "ALD range",
        "MASLD alcohol range",
        "MetALD range",
        "ALD range"
    ],
    default="Unknown"
)

print("Alcohol Classification:")
print(masld_data["alcohol_category"].value_counts(dropna=False))

print("\nAlcohol Classification by Sex:")
display(pd.crosstab(
    masld_data["RIAGENDR"],
    masld_data["alcohol_category"]
))

Alcohol Classification:
alcohol_category
MASLD alcohol range    3577
Unknown                 689
MetALD range            179
ALD range                64
Name: count, dtype: int64

Alcohol Classification by Sex:


alcohol_category,ALD range,MASLD alcohol range,MetALD range,Unknown
RIAGENDR,,,,
1.0,53,1830,108,241
2.0,11,1747,71,448


In [ ]:
# ------------------ Stage 9: MASLD Target Construction ------------------

# Step 1: Define hepatic steatosis using CAP threshold
masld_data["steatosis_flag"] = (
    masld_data["LUXCAPM"] >= 302
).astype(int)

# Step 2: Identify whether at least one metabolic risk factor is present
risk_cols = [
    "bmi_risk",
    "waist_risk",
    "glycemic_risk",
    "bp_risk",
    "triglyceride_risk",
    "hdl_risk"
]

risk_present = (masld_data[risk_cols] == 1).any(axis=1)

# A participant is confirmed as having no metabolic risk
# only when all six indicators are observed and negative
risk_absent = (masld_data[risk_cols] == 0).all(axis=1)

masld_data["metabolic_risk"] = np.select(
    [risk_present, risk_absent],
    [1, 0],
    default=np.nan
)

# Step 3: Construct the operational MASLD target
alcohol_eligible = (
    masld_data["alcohol_category"] == "MASLD alcohol range"
)

masld_data["MASLD"] = np.nan

# No steatosis means the MASLD target is negative
masld_data.loc[
    alcohol_eligible & (masld_data["steatosis_flag"] == 0),
    "MASLD"
] = 0

# Steatosis + at least one metabolic risk factor means positive
masld_data.loc[
    alcohol_eligible &
    (masld_data["steatosis_flag"] == 1) &
    (masld_data["metabolic_risk"] == 1),
    "MASLD"
] = 1

# Step 4: Display target distribution
print("Metabolic Risk Distribution:")
print(masld_data["metabolic_risk"].value_counts(dropna=False))

print("\nSteatosis Distribution:")
print(masld_data["steatosis_flag"].value_counts())

print("\nFinal MASLD Target Distribution:")
print(masld_data["MASLD"].value_counts(dropna=False).sort_index())

print("\nFinal Target Percentages:")
print(
    (masld_data["MASLD"].value_counts(normalize=True) * 100)
    .round(2)
)

print("\nEligible Modelling Participants:",
      masld_data["MASLD"].notna().sum())

print("Excluded / Unlabelled Participants:",
      masld_data["MASLD"].isna().sum())

Metabolic Risk Distribution:
metabolic_risk
1.0    4186
NaN     214
0.0     109
Name: count, dtype: int64

Steatosis Distribution:
steatosis_flag
0    3222
1    1287
Name: count, dtype: int64

Final MASLD Target Distribution:
MASLD
0.0    2543
1.0    1033
NaN     933
Name: count, dtype: int64

Final Target Percentages:
MASLD
0.0    71.11
1.0    28.89
Name: proportion, dtype: float64

Eligible Modelling Participants: 3576
Excluded / Unlabelled Participants: 933


# Stage 10: Final Dataset Preparation

In [ ]:
# ------------------ Stage 10: Final Dataset Preparation ------------------

# Keep participants with a defined MASLD target
model_data = masld_data[
    masld_data["MASLD"].notna()
].copy()

# Convert target to integer
model_data["MASLD"] = model_data["MASLD"].astype(int)

# Inspect target distribution
print("Model Dataset Shape:", model_data.shape)

print("\nTarget Distribution:")
print(model_data["MASLD"].value_counts().sort_index())

print("\nTarget Proportion:")
print(
    (model_data["MASLD"].value_counts(normalize=True) * 100)
    .round(2)
)

# Check participant uniqueness
print("\nUnique Participants:", model_data["SEQN"].nunique())
print("Duplicate Participants:", model_data["SEQN"].duplicated().sum())

Model Dataset Shape: (3576, 139)

Target Distribution:
MASLD
0    2543
1    1033
Name: count, dtype: int64

Target Proportion:
MASLD
0    71.11
1    28.89
Name: proportion, dtype: float64

Unique Participants: 3576
Duplicate Participants: 0


In [ ]:
# ------------------ Stage 10: Predictor Inspection ------------------

excluded_cols = [
    "SEQN",
    "MASLD",
    "LUXCAPM",
    "steatosis_flag",
    "metabolic_risk",
    "bmi_risk",
    "waist_risk",
    "glycemic_risk",
    "bp_risk",
    "triglyceride_risk",
    "hdl_risk",
    "alcohol_g_week",
    "alcohol_eligible",
    "alcohol_category"
]

candidate_features = [
    col for col in model_data.columns
    if col not in excluded_cols
]

print("Total candidate features:", len(candidate_features))

print("\nCandidate Features:")
for col in candidate_features:
    print(col)

Total candidate features: 125

Candidate Features:
RIDAGEYR
RIAGENDR
RIDRETH1
LUAXSTAT
LUARXNC
LUARXND
LUARXIN
LUAPNME
LUANMVGP
LUANMTGP
LUATECH
LUXSMED
LUXSIQR
LUXSIQRM
LUXCPIQR
BMDSTATS
BMXWT
BMIWT
BMXRECUM
BMIRECUM
BMXHEAD
BMIHEAD
BMXHT
BMIHT
BMXBMI
BMXLEG
BMILEG
BMXARML
BMIARML
BMXARMC
BMIARMC
BMXWAIST
BMIWAIST
BMXHIP
BMIHIP
DIQ010
DID040
DIQ160
DIQ170
DIQ172
DIQ175A
DIQ175B
DIQ175C
DIQ175D
DIQ175E
DIQ175F
DIQ175G
DIQ175H
DIQ175I
DIQ175J
DIQ175K
DIQ175L
DIQ175M
DIQ175N
DIQ175O
DIQ175P
DIQ175Q
DIQ175R
DIQ175S
DIQ175T
DIQ175U
DIQ175V
DIQ175W
DIQ175X
DIQ180
DIQ050
DID060
DIQ060U
DIQ070
DIQ230
DIQ240
DID250
DID260
DIQ260U
DIQ275
DIQ280
DIQ291
DIQ300S
DIQ300D
DID310S
DID310D
DID320
DID330
DID341
DID350
DIQ350U
DIQ360
DIQ080
BPQ020
BPQ030
BPD035
BPQ040A
BPQ050A
BPQ080
BPQ060
BPQ070
BPQ090D
BPQ100D
ALQ111
ALQ121
ALQ130
ALQ142
ALQ270
ALQ280
ALQ290
ALQ151
ALQ170
LBXGH
WTSAF2YR_x
LBXGLU
LBDGLUSI
LBDHDD
LBDHDDSI
WTSAF2YR_y
LBXTR
LBDTRSI
LBDLDL
LBDLDLSI
LBDLDLM
LBDLDMSI
LBDLDLN
LBDLDNSI
mean_s

In [ ]:
# ------------------ Stage 10: Final Feature Selection ------------------

selected_features = [
    "RIDAGEYR",       # Age
    "RIAGENDR",       # Sex
    "RIDRETH3",       # Detailed race/ethnicity
    "LUXSMED"         # Liver stiffness measurement
]

X = model_data[selected_features].copy()
y = model_data["MASLD"].copy()

print("Selected Features:", selected_features)
print("\nFeature Matrix Shape:", X.shape)
print("Target Shape:", y.shape)

print("\nFeature Missing Values:")
print(X.isna().sum())

print("\nTarget Distribution:")
print(y.value_counts())

Selected Features: ['RIDAGEYR', 'RIAGENDR', 'RIDRETH3', 'LUXSMED']

Feature Matrix Shape: (3576, 4)
Target Shape: (3576,)

Feature Missing Values:
RIDAGEYR    0
RIAGENDR    0
RIDRETH3    0
LUXSMED     0
dtype: int64

Target Distribution:
MASLD
0    2543
1    1033
Name: count, dtype: int64


# Stage 11: Train-Test Split

In [ ]:
# ------------------ Stage 11: Train-Test Split ------------------

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Features:", X_train.shape)
print("Testing Features:", X_test.shape)

print("\nTraining Target Distribution:")
print(y_train.value_counts(normalize=True).round(3))

print("\nTesting Target Distribution:")
print(y_test.value_counts(normalize=True).round(3))

Training Features: (2860, 4)
Testing Features: (716, 4)

Training Target Distribution:
MASLD
0    0.711
1    0.289
Name: proportion, dtype: float64

Testing Target Distribution:
MASLD
0    0.711
1    0.289
Name: proportion, dtype: float64


# Stage 12: Data Preprocessing

In [ ]:
# ------------------ Stage 12: Data Preprocessing ------------------

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

numeric_features = ["RIDAGEYR", "LUXSMED"]
categorical_features = ["RIAGENDR", "RIDRETH3"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

# Fit only on training data
X_train_processed = preprocessor.fit_transform(X_train)

# Transform test data using the fitted training preprocessor
X_test_processed = preprocessor.transform(X_test)

print("Processed Training Shape:", X_train_processed.shape)
print("Processed Testing Shape:", X_test_processed.shape)

Processed Training Shape: (2860, 10)
Processed Testing Shape: (716, 10)


# Stage 13: Model Training

In [ ]:
# ------------------ Stage 13: Model Training ------------------

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Initialize classification models
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        eval_metric="logloss",
        random_state=42
    )
}

# Train all models
for name, model in models.items():
    model.fit(X_train_processed, y_train)
    print(f"{name} training completed.")

Logistic Regression training completed.
Random Forest training completed.
XGBoost training completed.


# Stage 14: Model Evaluation

In [ ]:
# ------------------ Stage 14: Model Evaluation ------------------

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

results = []

for name, model in models.items():

    # Generate predictions
    y_pred = model.predict(X_test_processed)
    y_prob = model.predict_proba(X_test_processed)[:, 1]

    # Calculate evaluation metrics
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

    print(f"\n{name}")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))

# Compare model performance
results_df = pd.DataFrame(results)

print("\nModel Comparison:")
display(results_df.round(4))


Logistic Regression
Confusion Matrix:
[[309 200]
 [ 78 129]]

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.61      0.69       509
           1       0.39      0.62      0.48       207

    accuracy                           0.61       716
   macro avg       0.60      0.62      0.59       716
weighted avg       0.68      0.61      0.63       716


Random Forest
Confusion Matrix:
[[413  96]
 [143  64]]

Classification Report:
              precision    recall  f1-score   support

           0       0.74      0.81      0.78       509
           1       0.40      0.31      0.35       207

    accuracy                           0.67       716
   macro avg       0.57      0.56      0.56       716
weighted avg       0.64      0.67      0.65       716


XGBoost
Confusion Matrix:
[[459  50]
 [150  57]]

Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.90      0.82       5

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Logistic Regression,0.6117,0.3921,0.6232,0.4813,0.6762
1,Random Forest,0.6662,0.4000,0.3092,0.3488,0.6550
2,XGBoost,0.7207,0.5327,0.2754,0.3631,0.6878


# Stage 15: Threshold Analysis

In [ ]:
# ------------------ Stage 15: Threshold Analysis ------------------

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Get positive-class probabilities
lr_model = models["Logistic Regression"]
y_prob_lr = lr_model.predict_proba(X_test_processed)[:, 1]

thresholds = [0.30, 0.40, 0.50, 0.60, 0.70]

threshold_results = []

for threshold in thresholds:

    y_pred_threshold = (
        y_prob_lr >= threshold
    ).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_test, y_pred_threshold),
        "Precision": precision_score(
            y_test, y_pred_threshold, zero_division=0
        ),
        "Recall": recall_score(
            y_test, y_pred_threshold, zero_division=0
        ),
        "F1-Score": f1_score(
            y_test, y_pred_threshold, zero_division=0
        )
    })

threshold_df = pd.DataFrame(threshold_results)

print("Logistic Regression Threshold Analysis:")
display(threshold_df.round(4))

Logistic Regression Threshold Analysis:


,Threshold,Accuracy,Precision,Recall,F1-Score
0,0.3,0.3603,0.3090,0.9807,0.4699
1,0.4,0.4888,0.3456,0.8599,0.4931
2,0.5,0.6117,0.3921,0.6232,0.4813
3,0.6,0.7109,0.5000,0.3382,0.4035
4,0.7,0.7165,0.5345,0.1498,0.2340


# Stage 16: Model Interpretation

In [ ]:
# ------------------ Stage 16: Logistic Regression Coefficients ------------------

# Get transformed feature names
feature_names = preprocessor.get_feature_names_out()

# Extract Logistic Regression coefficients
lr_model = models["Logistic Regression"]

coefficient_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": lr_model.coef_[0]
})

coefficient_df["Absolute_Importance"] = (
    coefficient_df["Coefficient"].abs()
)

coefficient_df = coefficient_df.sort_values(
    by="Absolute_Importance",
    ascending=False
)

print("Logistic Regression Feature Coefficients:")
display(coefficient_df.round(4))

Logistic Regression Feature Coefficients:


,Feature,Coefficient,Absolute_Importance
1,num__LUXSMED,0.7265,0.7265
4,cat__RIDRETH3_1.0,0.5452,0.5452
7,cat__RIDRETH3_4.0,-0.3886,0.3886
0,num__RIDAGEYR,0.2717,0.2717
3,cat__RIAGENDR_2.0,-0.2158,0.2158
2,cat__RIAGENDR_1.0,0.2011,0.2011
9,cat__RIDRETH3_7.0,-0.1386,0.1386
8,cat__RIDRETH3_6.0,-0.0744,0.0744
6,cat__RIDRETH3_3.0,0.0301,0.0301
5,cat__RIDRETH3_2.0,0.0117,0.0117


# Stage 17: Final Results and Conclusion

In [ ]:
# ------------------ Stage 17: Final Results and Conclusion ------------------

print("NAFLD / MASLD MODEL SUMMARY")

print("\n1. Dataset")
print("Eligible participants:", len(model_data))
print("Training participants:", len(X_train))
print("Testing participants:", len(X_test))

print("\n2. Target Distribution")
print(model_data["MASLD"].value_counts())

print("\n3. Model Performance")
display(results_df.round(4))

print("\n4. Best Model by Recall")
print(
    results_df.loc[
        results_df["Recall"].idxmax()
    ]
)

print("\n5. Best Model by ROC-AUC")
print(
    results_df.loc[
        results_df["ROC-AUC"].idxmax()
    ]
)

print("\n6. Best F1-Score Threshold (Exploratory)")
display(
    threshold_df.loc[
        threshold_df["F1-Score"].idxmax()
    ]
)

NAFLD / MASLD MODEL SUMMARY

1. Dataset
Eligible participants: 3576
Training participants: 2860
Testing participants: 716

2. Target Distribution
MASLD
0    2543
1    1033
Name: count, dtype: int64

3. Model Performance


,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Logistic Regression,0.6117,0.3921,0.6232,0.4813,0.6762
1,Random Forest,0.6662,0.4000,0.3092,0.3488,0.6550
2,XGBoost,0.7207,0.5327,0.2754,0.3631,0.6878



4. Best Model by Recall
Model        Logistic Regression
Accuracy                0.611732
Precision               0.392097
Recall                  0.623188
F1-Score                0.481343
ROC-AUC                 0.676162
Name: 0, dtype: object

5. Best Model by ROC-AUC
Model         XGBoost
Accuracy      0.72067
Precision     0.53271
Recall       0.275362
F1-Score     0.363057
ROC-AUC      0.687817
Name: 2, dtype: object

6. Best F1-Score Threshold (Exploratory)


,1
Threshold,0.400000
Accuracy,0.488827
Precision,0.345631
Recall,0.859903
F1-Score,0.493075


### Final Results and Conclusion

- Analysed NHANES 2017–2018 data for NAFLD/MASLD classification.
- Integrated demographic, liver, anthropometric, metabolic, laboratory and questionnaire datasets.
- Constructed an operational MASLD target using CAP ≥302 dB/m, at least one cardiometabolic risk factor and approximate alcohol eligibility criteria.
- Obtained **3,576 labelled participants**:
  - MASLD negative: 2,543
  - MASLD positive: 1,033
- Performed an 80:20 stratified train-test split:
  - Training: 2,860 participants
  - Testing: 716 participants
- Applied StandardScaler to numerical features and OneHotEncoder to categorical features.
- Trained and evaluated three classification models:
  - Logistic Regression
  - Random Forest
  - XGBoost
- **Logistic Regression achieved the highest recall (62.32%) and F1-score (48.13%)**, making it the preferred initial screening candidate.
- **XGBoost achieved the highest accuracy (72.07%), precision (53.27%) and ROC-AUC (0.6878)**.
- Conducted exploratory threshold analysis for Logistic Regression; a threshold of 0.40 achieved the highest F1-score (49.31%) and recall (85.99%).
- Identified liver stiffness (`LUXSMED`) as the strongest predictor according to Logistic Regression coefficients.
- Observed modest predictive discrimination across all three models.

### Limitations

- The MASLD target represents an operational research classification, not a confirmed clinical diagnosis.
- Alcohol consumption was estimated using categorical questionnaire responses.
- Fasting glucose and triglyceride measurements were available only for a subsample.
- The test-set threshold analysis requires independent validation.
- NHANES survey weights were not incorporated.
- Liver stiffness was measured during the same examination as CAP, which may influence predictive performance.

### Conclusion

- Logistic Regression provided the strongest recall-oriented baseline, while XGBoost achieved the highest overall accuracy and ROC-AUC.
- The analysis demonstrates a machine-learning approach for identifying CAP-defined hepatic steatosis accompanied by cardiometabolic risk factors.
- Further validation, feature evaluation and testing on independent datasets are required before clinical application.